# Qwen3.8-27B → DeepSeek Harness

This notebook uses the outbound-only Growing-Trader Supabase relay. No Cloudflare, ngrok, localhost.run, reverse proxy, or public Colab port is used.

In Colab **Secrets** (key icon), add `QWEN_RELAY_SECRET` and enable **Notebook access**.

For automatic Oracle VM wake-up, also add `ORACLE_WAKE_GITHUB_TOKEN` and enable **Notebook access**. Use a fine-grained GitHub token restricted to the relay repository with **Actions: Read and write** permission. Colab uses it only to dispatch the existing wake workflow; cloud credentials remain in GitHub Actions.

When Section 2 starts, Colab immediately creates a short heartbeat lease in Supabase. This keeps the shared Oracle VM online while Qwen is still installing/downloading/loading. The public API does **not** report Qwen ready until vLLM is actually ready.

For a network/API test with **zero GPU usage**, run **Section 1** and then **Section 3** only. For the real model, run **Section 1** and then **Section 2** on an A100 80 GB runtime. Do not run Sections 2 and 3 at the same time.


## Section 1 — Install / update GitHub package
Run this for both the real worker and the no-GPU test worker.


In [ ]:
%pip install -q --upgrade --force-reinstall "git+https://github.com/Logan17de/All-testing.git#subdirectory=llm"


## Optional — capture BEFORE benchmark
If the old BF16 vLLM server is **still running in this same Colab runtime**, run this once before Section 2. It measures localhost TTFT and decode tokens/sec and saves the result under `/content/qwen_benchmark_results.json`. Section 2 also auto-detects and benchmarks a legacy running server before replacing it, so this cell is optional.


In [ ]:
import importlib
import qwen3_8_27b_supabase_colab_fast as qwen_fast
importlib.reload(qwen_fast)
qwen_fast.benchmark_running_server("before")


## Section 2 — REAL Qwen3.8-27B optimized worker (A100 80 GB)
This uses the official `Qwen/Qwen3.8-27B-FP8` weights, native 262,144 context, FP8 KV cache, native MTP speculative decoding with 3 draft tokens, prefix caching, chunked prefill, torch.compile/CUDA Graphs, and a single-user scheduling profile. On A100, FP8 KV is not compatible with FlashAttention 2, so the profile uses FlashInfer as the fast compatible attention backend. The local benchmark runs before the worker is advertised ready and prints TTFT + output tok/s; if a BEFORE result exists it also prints the before/after speedup.


In [ ]:
import importlib
import qwen3_8_27b_supabase_colab_fast as qwen_worker
importlib.reload(qwen_worker)
qwen_worker.main()


## Section 3 — TESTING: API/relay only (NO GPU)
Use a normal CPU Colab runtime. Run **Section 1**, skip Section 2, then run this section. No Torch, vLLM, CUDA, Hugging Face model, or GPU is used. Every request received from Harness is decoded through the real Growing-Trader relay and returned as the OpenAI-compatible assistant response **`succeed`**. Leave this cell running while testing Harness. This section can also wake Oracle automatically when `ORACLE_WAKE_GITHUB_TOKEN` is configured.


In [ ]:
import importlib
import qwen_supabase_test_worker as relay_test
importlib.reload(relay_test)
relay_test.main()
